# SparkyLLM agent eval — 100-prompt DeepSeek-judged trace run

Runs `local_agent.AgentRunner` over 100 bucketed prompts, judges each trace with DeepSeek (LLM-as-judge), and streams every trace + score to a JSONL file for future inspection.

Designed for Colab (T4 GPU). Loads the tool-use checkpoint `gpt_medium_tools_best.pth` via the same Drive layout as `local_agent/colab_deploy.ipynb`.

**Prereqs in Colab:**
1. Runtime → T4 GPU
2. `DEEPSEEK_API_KEY` in Colab Secrets (Settings → Secrets)
3. Drive has `MyDrive/sparkyllm/checkpoints/gpt_medium_tools_best.pth` and `MyDrive/sparkyllm/datasets_pretrain/tokenizer_out/tokenizer.json`

Output JSONL (one trace per line) lands at `MyDrive/sparkyllm/agent_evals/run_<UTC-timestamp>.jsonl` on Colab, or `test/agent_eval_runs/` locally.

## 1. Install dependencies

`torch` is preinstalled in Colab. Everything else is small.

In [ ]:
!pip install -q openai tokenizers tqdm pandas

## 2. Clone or pull sparkyllm

Mirrors `colab_deploy.ipynb`. Locally, assumes this notebook lives at `<repo>/test/test_agent.ipynb`.

In [ ]:
import os, sys, subprocess

IS_COLAB = "google.colab" in sys.modules

if IS_COLAB:
    REPO_URL = "https://github.com/ajencinas/sparkyllm.git"
    REPO_DIR = "/content/sparkyllm"
    if not os.path.exists(REPO_DIR):
        print("Cloning repo...")
        subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
    else:
        print("Repo exists, pulling latest...")
        subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
else:
    REPO_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
    print(f"Local run. REPO_DIR = {REPO_DIR}")

os.chdir(REPO_DIR)
print("CWD:", os.getcwd())

## 3. Mount Drive + stage weights (Colab only)

Copies the tool-use checkpoint + tokenizer into `local_test/weights/` — the layout `AgentRunner` expects. Skip if running locally with weights already staged.

In [ ]:
import shutil

WEIGHTS_DIR = os.path.join(REPO_DIR, "local_test", "weights")
os.makedirs(WEIGHTS_DIR, exist_ok=True)

if IS_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    SPARKYLLM_DRIVE = "/content/drive/MyDrive/sparkyllm"
    files_to_copy = [
        ("checkpoints/gpt_medium_tools_best.pth", "gpt_medium_tools_best.pth"),
        ("datasets_pretrain/tokenizer_out/tokenizer.json", "tokenizer.json"),
    ]
    for src_rel, dst_name in files_to_copy:
        src = os.path.join(SPARKYLLM_DRIVE, src_rel)
        dst = os.path.join(WEIGHTS_DIR, dst_name)
        if not os.path.exists(src):
            print(f"  MISSING on Drive: {src}")
            continue
        if os.path.exists(dst):
            print(f"  already in place: {dst_name}")
            continue
        print(f"  copying {dst_name}...")
        shutil.copy2(src, dst)

print()
print("Weights folder:")
for f in sorted(os.listdir(WEIGHTS_DIR)):
    sz = os.path.getsize(os.path.join(WEIGHTS_DIR, f)) / 1024 / 1024
    print(f"  {f}  ({sz:.1f} MB)")

## 3b. Brave Search API key (for better web_search relevance)

`web_search` prefers Brave over Wikipedia/DDG when `BRAVE_API_KEY` is set. Free tier is 2000 queries/month. Skip if you haven't added the key — will fall back to Wikipedia + strict DDG.

In [ ]:
# Load BRAVE_API_KEY from Colab Secrets (or env var locally).
# Must run BEFORE cell 4 imports `tools` — that's when the key is read.
if IS_COLAB:
    try:
        from google.colab import userdata
        key = userdata.get("BRAVE_API_KEY")
        if key:
            os.environ["BRAVE_API_KEY"] = key
            print("Brave API key loaded from Colab Secrets")
        else:
            print("BRAVE_API_KEY not found in Colab Secrets — will fall back to Wikipedia + strict DDG")
    except Exception as e:
        print(f"Could not read Colab Secrets ({e}) — will fall back to Wikipedia + strict DDG")
else:
    if os.environ.get("BRAVE_API_KEY"):
        print("BRAVE_API_KEY present in env")
    else:
        print("BRAVE_API_KEY not set — tools.py will try .env, else fall back to Wikipedia + strict DDG")

## 4. Load model + agent runner

In [ ]:
sys.path.insert(0, os.path.join(REPO_DIR, "local_test"))
sys.path.insert(0, os.path.join(REPO_DIR, "local_agent"))

from sparky_model import detect_device, load_model, load_tokenizer, vocab_size_for  # type: ignore
from agent import AgentRunner, AgentResult  # type: ignore
from tools import TOOLS  # type: ignore

TOKENIZER_PATH = os.path.join(WEIGHTS_DIR, "tokenizer.json")
CHECKPOINT_PATH = os.path.join(WEIGHTS_DIR, "gpt_medium_tools_best.pth")

assert os.path.exists(TOKENIZER_PATH), f"Missing tokenizer at {TOKENIZER_PATH}"
assert os.path.exists(CHECKPOINT_PATH), f"Missing checkpoint at {CHECKPOINT_PATH}"

device = detect_device()
tokenizer = load_tokenizer(TOKENIZER_PATH)
vocab = vocab_size_for(tokenizer)
model = load_model(CHECKPOINT_PATH, vocab, device)

runner = AgentRunner(
    model, tokenizer, device,
    max_steps=3,
    temperature=0.7,
)
print(f"Loaded tools checkpoint on {device}. Tools available: {list(TOOLS)}")

## 5. DeepSeek judge client

Same pattern as `test/eval_pipeline.ipynb`: OpenAI SDK pointed at DeepSeek, JSON rubric with retries.

In [ ]:
import json, time as _time
from openai import OpenAI

if IS_COLAB:
    from google.colab import userdata
    DEEPSEEK_API_KEY = userdata.get("DEEPSEEK_API_KEY")
else:
    DEEPSEEK_API_KEY = os.environ.get("DEEPSEEK_API_KEY")
assert DEEPSEEK_API_KEY, "DEEPSEEK_API_KEY not found (Colab Secrets or env var)"

judge_client = OpenAI(api_key=DEEPSEEK_API_KEY, base_url="https://api.deepseek.com")
print("DeepSeek judge client ready")

JUDGE_SYSTEM = """You are evaluating a small (650M parameter) language model running a ReAct-style tool-use agent.

You will receive a user PROMPT, the agent's STEPS (Thought/Action/Input/Result sequence), and the agent's FINAL_ANSWER.

Available tools the agent can call:
- calculator  — arithmetic expressions like 2+3*4
- time        — returns current date/time, no input
- web_search  — factual lookup (DuckDuckGo/Wikipedia)
- none        — sentinel meaning "answer directly"

Score the agent on each criterion using a 1-5 scale (5 = excellent, 1 = failure):
- tool_choice: did the agent pick the right tool(s), or correctly skip tools?
- tool_input: were the tool inputs well-formed and appropriate? (5 if no tool was called.)
- final_answer_correctness: does FINAL_ANSWER correctly and fully answer the prompt?
- trace_coherence: are the Thoughts sensible and does the trace flow logically? Penalize loops, gibberish, or hallucinated tool outputs.

Reply with JSON only, no markdown fences, in this exact shape:
{\"tool_choice\": N, \"tool_input\": N, \"final_answer_correctness\": N, \"trace_coherence\": N, \"reason\": \"one short sentence\"}"""


def _format_steps(steps):
    if not steps:
        return "(no tool calls — direct answer)"
    parts = []
    for i, s in enumerate(steps, 1):
        result_preview = s.result[:300] + ("..." if len(s.result) > 300 else "")
        line = (
            f"Step {i}:\n"
            f"  Thought: {s.thought}\n"
            f"  Action: {s.action}\n"
            f"  Input: {s.input}\n"
            f"  Result: {result_preview}"
        )
        if s.error:
            line += f"\n  ERROR: {s.error}"
        parts.append(line)
    return "\n".join(parts)


def judge_trace(prompt, result, max_retries=3):
    user_msg = (
        f"PROMPT:\n{prompt}\n\n"
        f"STEPS:\n{_format_steps(result.steps)}\n\n"
        f"FINAL_ANSWER:\n{result.final_answer}\n\n"
        f"(truncated={result.truncated})"
    )
    last_err = None
    for attempt in range(max_retries):
        try:
            completion = judge_client.chat.completions.create(
                model="deepseek-chat",
                messages=[
                    {"role": "system", "content": JUDGE_SYSTEM},
                    {"role": "user", "content": user_msg},
                ],
                temperature=0.0,
                max_tokens=200,
            )
            raw = completion.choices[0].message.content.strip()
            if raw.startswith("```"):
                raw = raw.strip("`").strip()
                if raw.lower().startswith("json"):
                    raw = raw[4:].strip()
            scores = json.loads(raw)
            for key in ("tool_choice", "tool_input", "final_answer_correctness", "trace_coherence"):
                assert key in scores and 1 <= int(scores[key]) <= 5
                scores[key] = int(scores[key])
            scores.setdefault("reason", "")
            return scores
        except Exception as e:
            last_err = e
            if attempt < max_retries - 1:
                _time.sleep(1.5)
    return {
        "tool_choice": 0, "tool_input": 0,
        "final_answer_correctness": 0, "trace_coherence": 0,
        "reason": f"judge_error: {last_err}",
    }

print("judge_trace() ready")

## 6. Prompt bank — 100 bucketed prompts

| category | n | expected behavior |
|---|---|---|
| `calculator` | 25 | invoke `calculator` |
| `time`       | 15 | invoke `time` |
| `web_search` | 25 | invoke `web_search` |
| `multi_step` | 20 | 2+ distinct tools |
| `no_tool`    | 10 | direct `Final:` |
| `edge`       | 5  | graceful failure / judgment call |

In [ ]:
PROMPTS = [
    # --- calculator (25) ---
    {"category": "calculator", "prompt": "What is 42 * 3?"},
    {"category": "calculator", "prompt": "What's 17 + 28?"},
    {"category": "calculator", "prompt": "Calculate 100 - 37."},
    {"category": "calculator", "prompt": "What's 256 divided by 8?"},
    {"category": "calculator", "prompt": "Compute 2 to the power of 10."},
    {"category": "calculator", "prompt": "What is 15 times 15?"},
    {"category": "calculator", "prompt": "What's 999 + 1?"},
    {"category": "calculator", "prompt": "Calculate (12 + 8) * 3."},
    {"category": "calculator", "prompt": "What's 50 divided by 2, plus 10?"},
    {"category": "calculator", "prompt": "Compute 7 * 8 - 6."},
    {"category": "calculator", "prompt": "What's 144 divided by 12?"},
    {"category": "calculator", "prompt": "Calculate 3 to the power of 5."},
    {"category": "calculator", "prompt": "What is 1000 - 753?"},
    {"category": "calculator", "prompt": "Compute 25 * 4."},
    {"category": "calculator", "prompt": "What's 9 + 10?"},
    {"category": "calculator", "prompt": "Calculate 81 / 9."},
    {"category": "calculator", "prompt": "What is (5 + 3) * (7 - 2)?"},
    {"category": "calculator", "prompt": "Compute 2 + 2 * 2."},
    {"category": "calculator", "prompt": "What's 17 times 23?"},
    {"category": "calculator", "prompt": "Calculate 1024 divided by 32."},
    {"category": "calculator", "prompt": "What's 13 squared?"},
    {"category": "calculator", "prompt": "Compute 99 * 99."},
    {"category": "calculator", "prompt": "What is 47 + 58 + 93?"},
    {"category": "calculator", "prompt": "Calculate 100 / 7."},
    {"category": "calculator", "prompt": "What's 2 to the 16th power?"},

    # --- time (15) ---
    {"category": "time", "prompt": "What time is it?"},
    {"category": "time", "prompt": "What is today's date?"},
    {"category": "time", "prompt": "Tell me the current time."},
    {"category": "time", "prompt": "What's the date and time right now?"},
    {"category": "time", "prompt": "Give me the current timestamp."},
    {"category": "time", "prompt": "What day is it today?"},
    {"category": "time", "prompt": "Tell me the time please."},
    {"category": "time", "prompt": "What is the current date?"},
    {"category": "time", "prompt": "What's today's date?"},
    {"category": "time", "prompt": "What is the time right now?"},
    {"category": "time", "prompt": "What time is it currently?"},
    {"category": "time", "prompt": "Please tell me the time."},
    {"category": "time", "prompt": "What is the date today?"},
    {"category": "time", "prompt": "Give me the current date and time."},
    {"category": "time", "prompt": "Current time please."},

    # --- web_search (25) ---
    {"category": "web_search", "prompt": "Who was Albert Einstein?"},
    {"category": "web_search", "prompt": "What is the capital of France?"},
    {"category": "web_search", "prompt": "Who invented the telephone?"},
    {"category": "web_search", "prompt": "What is photosynthesis?"},
    {"category": "web_search", "prompt": "Who wrote Hamlet?"},
    {"category": "web_search", "prompt": "What is the Eiffel Tower?"},
    {"category": "web_search", "prompt": "Who was Isaac Newton?"},
    {"category": "web_search", "prompt": "What is DNA?"},
    {"category": "web_search", "prompt": "Who was the first president of the United States?"},
    {"category": "web_search", "prompt": "What is the largest ocean on Earth?"},
    {"category": "web_search", "prompt": "Who painted the Mona Lisa?"},
    {"category": "web_search", "prompt": "What is the speed of light?"},
    {"category": "web_search", "prompt": "Who was Nikola Tesla?"},
    {"category": "web_search", "prompt": "What is quantum mechanics?"},
    {"category": "web_search", "prompt": "Who developed the theory of relativity?"},
    {"category": "web_search", "prompt": "What is the Great Wall of China?"},
    {"category": "web_search", "prompt": "Who was Marie Curie?"},
    {"category": "web_search", "prompt": "What is the tallest mountain in the world?"},
    {"category": "web_search", "prompt": "Who discovered penicillin?"},
    {"category": "web_search", "prompt": "What is the Mariana Trench?"},
    {"category": "web_search", "prompt": "Who was Leonardo da Vinci?"},
    {"category": "web_search", "prompt": "What is a black hole?"},
    {"category": "web_search", "prompt": "Who composed the Ninth Symphony?"},
    {"category": "web_search", "prompt": "What is the population of Tokyo?"},
    {"category": "web_search", "prompt": "Who founded Microsoft?"},

    # --- multi_step (20) ---
    {"category": "multi_step", "prompt": "Add 5 plus 5, and then tell me the time."},
    {"category": "multi_step", "prompt": "What is 12 times 12, and what time is it now?"},
    {"category": "multi_step", "prompt": "Tell me today's date and calculate 100 divided by 4."},
    {"category": "multi_step", "prompt": "Who was Einstein, and what is 9 times 9?"},
    {"category": "multi_step", "prompt": "What is 2 to the 8th power and what's the current time?"},
    {"category": "multi_step", "prompt": "Calculate 7 times 6 then tell me the date."},
    {"category": "multi_step", "prompt": "What's 50 plus 50 and who invented the telephone?"},
    {"category": "multi_step", "prompt": "Tell me the time and then compute 1000 minus 1."},
    {"category": "multi_step", "prompt": "What is 81 divided by 9 and what's today's date?"},
    {"category": "multi_step", "prompt": "Compute 15 times 3 and look up Marie Curie."},
    {"category": "multi_step", "prompt": "What's 11 times 11 and the current time?"},
    {"category": "multi_step", "prompt": "Calculate 64 divided by 8 then tell me the date."},
    {"category": "multi_step", "prompt": "What is 3 to the 4th power and what is photosynthesis?"},
    {"category": "multi_step", "prompt": "Tell me today's date and calculate 7 plus 8."},
    {"category": "multi_step", "prompt": "What is 144 divided by 12 and who wrote Hamlet?"},
    {"category": "multi_step", "prompt": "Compute 25 times 4 and then give me the time."},
    {"category": "multi_step", "prompt": "What is 2 plus 2 and what's the capital of France?"},
    {"category": "multi_step", "prompt": "Who invented the telephone and what's 6 times 7?"},
    {"category": "multi_step", "prompt": "What's the date and what is 18 minus 9?"},
    {"category": "multi_step", "prompt": "Calculate 5 times 4 times 3 times 2, then tell me the time."},

    # --- no_tool (10) ---
    {"category": "no_tool", "prompt": "Hi, how are you?"},
    {"category": "no_tool", "prompt": "Hello!"},
    {"category": "no_tool", "prompt": "What's your name?"},
    {"category": "no_tool", "prompt": "Tell me a joke."},
    {"category": "no_tool", "prompt": "Good morning."},
    {"category": "no_tool", "prompt": "Thanks!"},
    {"category": "no_tool", "prompt": "How's it going?"},
    {"category": "no_tool", "prompt": "Can you help me?"},
    {"category": "no_tool", "prompt": "Nice to meet you."},
    {"category": "no_tool", "prompt": "Say hello."},

    # --- edge (5) ---
    {"category": "edge", "prompt": "What is 1 divided by 0?"},
    {"category": "edge", "prompt": "Calculate abcdefg."},
    {"category": "edge", "prompt": "Who is the best football player in the world right now?"},
    {"category": "edge", "prompt": "What is the meaning of life?"},
    {"category": "edge", "prompt": "help"},
]

for i, p in enumerate(PROMPTS):
    p["id"] = i

from collections import Counter
print(f"Total prompts: {len(PROMPTS)}")
print("By category:", dict(Counter(p["category"] for p in PROMPTS)))

## 7. Run + judge + persist to JSONL

Streams one record per line to `agent_evals/run_<UTC-timestamp>.jsonl` (Drive on Colab, in-repo otherwise). Each write is flushed so partial progress survives a kernel crash. Exceptions in `run_turn` or the judge are captured into the record — they don't abort the run.

In [ ]:
from datetime import datetime, timezone
from dataclasses import asdict
from tqdm.auto import tqdm

if IS_COLAB:
    OUT_DIR = "/content/drive/MyDrive/sparkyllm/agent_evals"
else:
    OUT_DIR = os.path.join(REPO_DIR, "test", "agent_eval_runs")
os.makedirs(OUT_DIR, exist_ok=True)

ts = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%SZ")
OUT_PATH = os.path.join(OUT_DIR, f"run_{ts}.jsonl")
print(f"Writing traces to: {OUT_PATH}")


def _serialize_steps(steps):
    return [asdict(s) for s in steps]


records = []
with open(OUT_PATH, "w", encoding="utf-8") as fout:
    for item in tqdm(PROMPTS, desc="agent eval"):
        prompt = item["prompt"]
        rec = {
            "id": item["id"],
            "category": item["category"],
            "prompt": prompt,
            "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        }

        # --- 1) run the agent ---
        t0 = _time.time()
        try:
            result = runner.run_turn(prompt)
            rec["final_answer"] = result.final_answer
            rec["steps"] = _serialize_steps(result.steps)
            rec["raw_trace"] = result.raw_trace
            rec["truncated"] = result.truncated
            rec["agent_error"] = None
        except Exception as e:
            result = None
            rec["final_answer"] = ""
            rec["steps"] = []
            rec["raw_trace"] = ""
            rec["truncated"] = False
            rec["agent_error"] = f"{type(e).__name__}: {e}"
        rec["agent_latency_s"] = round(_time.time() - t0, 2)
        rec["tools_used"] = [s["action"] for s in rec["steps"]]

        # --- 2) judge with DeepSeek ---
        if result is not None:
            t1 = _time.time()
            scores = judge_trace(prompt, result)
            rec["judge_latency_s"] = round(_time.time() - t1, 2)
            rec.update(scores)
        else:
            rec.update({
                "tool_choice": 0, "tool_input": 0,
                "final_answer_correctness": 0, "trace_coherence": 0,
                "reason": "agent crashed", "judge_latency_s": 0.0,
            })

        fout.write(json.dumps(rec, ensure_ascii=False) + "\n")
        fout.flush()
        records.append(rec)

print(f"\nDone. Wrote {len(records)} records to {OUT_PATH}")

## 8. Summary

Aggregate scores overall and by category, then print the worst 10 traces for manual inspection.

In [ ]:
import pandas as pd

SCORE_COLS = ["tool_choice", "tool_input", "final_answer_correctness", "trace_coherence"]

df = pd.DataFrame(records)
df["avg_score"] = df[SCORE_COLS].mean(axis=1)

print("=== Overall means (1-5) ===")
print(df[SCORE_COLS + ["avg_score"]].mean().round(2).to_string())

print("\n=== By category ===")
print(
    df.groupby("category")[SCORE_COLS + ["avg_score"]]
      .mean()
      .round(2)
      .sort_values("avg_score")
      .to_string()
)

n_truncated = int(df["truncated"].sum()) if "truncated" in df.columns else 0
n_agent_errors = int(df["agent_error"].notna().sum())
n_judge_errors = int(df["reason"].str.startswith("judge_error").sum())
print(f"\ntruncated: {n_truncated} / {len(df)}")
print(f"agent errors: {n_agent_errors}")
print(f"judge errors: {n_judge_errors}")

print("\n=== Worst 10 by avg_score ===")
worst = df.nsmallest(10, "avg_score")[["id", "category", "prompt", "avg_score", "final_answer", "reason"]]
for _, row in worst.iterrows():
    print(f"\n[{row['id']:>3}] ({row['category']}) avg={row['avg_score']:.2f}")
    print(f"  prompt: {row['prompt']}")
    ans = (row['final_answer'] or '').replace('\n', ' ')
    print(f"  final:  {ans[:160]}{'...' if len(ans) > 160 else ''}")
    print(f"  judge:  {row['reason']}")

print(f"\nJSONL saved to: {OUT_PATH}")
print("Re-load any time with:  pd.read_json(path, lines=True)")